# 02 - Clean the complete JPS coastal dataset

Input: `data/raw/jps_coastal_erosion_full.csv` (144 original stations).
Output: `data/cleaned/jps_coastal_erosion_cleaned_144.csv`.

Run all cells from a fresh Python kernel. This notebook preserves the existing
144-station cleaning policy: retain Protected locations without inventing erosion
rates, preserve source total scores separately, calculate cleaned scores, and flag
erosion outliers without removing them. The older 125-station dataset is not used.


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Locate the project from either the repository root or its notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data" / "raw" / "jps_coastal_erosion_full.csv").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Open this notebook from inside FYP_Coastal_Erosion.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Step 1: Load the complete original JPS dataset.
input_path = PROJECT_ROOT / "data" / "raw" / "jps_coastal_erosion_full.csv"
df = pd.read_csv(input_path)
df_clean = df.copy()
print("Dataset shape:", df_clean.shape)
display(df_clean.head())


Dataset shape: (144, 14)


,No,Coastal_Cell,District,Area_Location,Longitude_DMS,Latitude_DMS,Longitude_DD,Latitude_DD,Annual_Erosion_Rate_m_yr,Physical_Erosion_Score,Economic_Score,Total_Score,ISMP_2022_Category,NCES_2015_Category
0,1,C1,Besut,North of Kampung Teluk Budu,"102°32'47.8""E","5°50'18.7""N",102.546611,5.838528,4.94,5.00,3.4,17.00,1,1
1,2,C1,Besut,Breakwater near Sungai Besut river mouth,"102°33'26.7""E","5°50'03.3""N",102.557417,5.834250,0.00,0.00,3.2,0.00,3,2
2,3,C1,Besut,Pantai Bukit Keluang,"102°36'17.9""E","5°48'08.2""N",102.604972,5.802278,1.92,2.00,2.3,4.60,3,2
3,4,C1,Besut,Pantai Bukit Keluang,"102°36'35.9""E","5°47'56.6""N",102.609972,5.799056,0.94,0.94,1.4,1.32,3,3
4,5,C1,Besut,Pantai Bukit Keluang,"102°36'40.1""E","5°47'51.6""N",102.611139,5.797667,5.28,5.00,1.0,5.00,3,3


In [2]:
# Step 2: Identify Protected records before numeric conversion.

df_clean["Is_Protected"] = (
    df_clean["Annual_Erosion_Rate_m_yr"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("protected")
).astype(int)

df_clean["Annual_Erosion_Rate_m_yr"] = pd.to_numeric(
    df_clean["Annual_Erosion_Rate_m_yr"],
    errors="coerce"
)

df_clean["Physical_Erosion_Score"] = pd.to_numeric(
    df_clean["Physical_Erosion_Score"],
    errors="coerce"
)

df_clean["Economic_Score"] = pd.to_numeric(
    df_clean["Economic_Score"],
    errors="coerce"
)

df_clean["Total_Score"] = pd.to_numeric(
    df_clean["Total_Score"],
    errors="coerce"
)

print("Protection status:")
print(df_clean["Is_Protected"].value_counts())
print("\nMissing numerical values:")
print(df_clean[["Annual_Erosion_Rate_m_yr", "Physical_Erosion_Score", "Economic_Score", "Total_Score"]].isnull().sum())


Protection status:
Is_Protected
0    135
1      9
Name: count, dtype: int64

Missing numerical values:
Annual_Erosion_Rate_m_yr    9
Physical_Erosion_Score      9
Economic_Score              0
Total_Score                 0
dtype: int64


In [3]:
# Step 3: Verify source station IDs, duplicates, and coordinates.
assert len(df_clean) == 144, "Expected the complete 144-station JPS dataset"
assert set(df_clean["No"]) == set(range(1, 145)), "Missing or unexpected JPS station IDs"
assert not df_clean["No"].duplicated().any(), "Duplicate JPS station IDs"
assert not df_clean.duplicated(["Latitude_DD", "Longitude_DD"]).any(), "Duplicate coordinates"
assert df_clean["Latitude_DD"].between(-90, 90).all(), "Invalid latitude"
assert df_clean["Longitude_DD"].between(-180, 180).all(), "Invalid longitude"

print("Original JPS stations:", len(df_clean))
print("Missing station numbers:", sorted(set(range(1, 145)) - set(df_clean["No"])))
print("Duplicate station numbers:", df_clean["No"].duplicated().sum())
print("Latitude range:", df_clean["Latitude_DD"].min(), "to", df_clean["Latitude_DD"].max())
print("Longitude range:", df_clean["Longitude_DD"].min(), "to", df_clean["Longitude_DD"].max())


Original JPS stations: 144
Missing station numbers: []
Duplicate station numbers: 0
Latitude range: 4.21125 to 5.838528
Longitude range: 102.546611 to 103.484222


## Cleaned score policy

Keep the reported score in `Total_Score` and `Original_Total_Score`. As in the
existing 144-station pipeline, calculate `Total_Score_Cleaned` from physical and
economic scores and retain a zero cleaned total for Protected locations. Display
differences for review rather than silently replacing the original source score.


In [4]:
# Step 4: Preserve source scores and calculate cleaned scores.

# Preserve original JPS total scores
df_clean["Original_Total_Score"] = df_clean["Total_Score"].copy()

# Calculate total scores
df_clean["Total_Score_Cleaned"] = (
    df_clean["Physical_Erosion_Score"] *
    df_clean["Economic_Score"]
).round(2)

# Protected locations retain their original zero total score
df_clean.loc[
    df_clean["Is_Protected"] == 1,
    "Total_Score_Cleaned"
] = 0.0

# Find significant score differences
mismatch = df_clean[
    (df_clean["Is_Protected"] == 0) &
    (
        abs(
            df_clean["Original_Total_Score"] -
            df_clean["Total_Score_Cleaned"]
        ) > 0.02
    )
]

print("Number of score mismatches:", len(mismatch))

print(mismatch[
    ["No", "Area_Location", "Original_Total_Score",
     "Total_Score_Cleaned"]
])


Number of score mismatches: 1
      No             Area_Location  Original_Total_Score  Total_Score_Cleaned
127  128  Petronas Kertih Refinery                   9.3                 15.5


In [5]:
# Step 5: Flag erosion-rate outliers without dropping stations.

# Select erosion rates from non-protected locations
erosion_data = df_clean.loc[
    df_clean["Is_Protected"] == 0,
    "Annual_Erosion_Rate_m_yr"
].dropna()

# Calculate IQR
Q1 = erosion_data.quantile(0.25)
Q3 = erosion_data.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
df_clean["Is_Erosion_Outlier"] = (
    (df_clean["Is_Protected"] == 0) &
    (
        (df_clean["Annual_Erosion_Rate_m_yr"] < lower_bound) |
        (df_clean["Annual_Erosion_Rate_m_yr"] > upper_bound)
    )
)

outliers = df_clean[df_clean["Is_Erosion_Outlier"]]

print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Number of outliers:", len(outliers))

print(outliers[
    ["No", "Area_Location", "Annual_Erosion_Rate_m_yr"]
])


Lower bound: -1.77
Upper bound: 5.53
Number of outliers: 10
      No                         Area_Location  Annual_Erosion_Rate_m_yr
16    17     North of Sungai Setiu river mouth                      7.32
17    18              Sungai Setiu river mouth                     15.52
18    19              Sungai Setiu river mouth                      7.09
19    20              Sungai Setiu river mouth                      7.52
20    21           Sand spit near Sungai Setiu                      8.76
21    22     West of Sungai Merang river mouth                     12.59
33    34                  Kampung Padang Maras                      7.85
34    35  Southeast of Kampung Pengkalan Maras                     10.18
127  128              Petronas Kertih Refinery                      8.19
143  144               Kampung Geliga Cik Akob                     10.56


In [6]:
# Step 6: Verify the cleaned dataset.

print("=== FINAL DATA CLEANING CHECK ===")

print("Dataset shape:", df_clean.shape)

print("\nProtected locations:")
print(df_clean["Is_Protected"].value_counts())

print("\nMissing values:")
print(df_clean[
    ["Annual_Erosion_Rate_m_yr",
     "Physical_Erosion_Score",
     "Economic_Score",
     "Total_Score_Cleaned"]
].isnull().sum())

print("\nDuplicate records:", df_clean.duplicated().sum())

print("\nOutliers:", df_clean["Is_Erosion_Outlier"].sum())

print("\nNon-protected missing values:")
print(df_clean.loc[
    df_clean["Is_Protected"] == 0,
    ["Annual_Erosion_Rate_m_yr",
     "Physical_Erosion_Score",
     "Economic_Score"]
].isnull().sum())

required_features = ["Annual_Erosion_Rate_m_yr", "Physical_Erosion_Score", "Economic_Score"]
protected_mask = df_clean["Is_Protected"].eq(1)
assert df_clean.loc[~protected_mask, required_features].notna().all().all()
assert df_clean.loc[protected_mask, "Annual_Erosion_Rate_m_yr"].isna().all()
assert df_clean.loc[protected_mask, "Physical_Erosion_Score"].isna().all()
assert df_clean["Total_Score_Cleaned"].notna().all()


=== FINAL DATA CLEANING CHECK ===
Dataset shape: (144, 18)

Protected locations:
Is_Protected
0    135
1      9
Name: count, dtype: int64

Missing values:
Annual_Erosion_Rate_m_yr    9
Physical_Erosion_Score      9
Economic_Score              0
Total_Score_Cleaned         0
dtype: int64

Duplicate records: 0

Outliers: 10

Non-protected missing values:
Annual_Erosion_Rate_m_yr    0
Physical_Erosion_Score      0
Economic_Score              0
dtype: int64


In [7]:
# Step 7: Save and verify the cleaned 144-station dataset.
output_path = PROJECT_ROOT / "data" / "cleaned" / "jps_coastal_erosion_cleaned_144.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False)
df_check = pd.read_csv(output_path)
assert len(df_check) == len(df_clean)
print("Cleaned dataset saved successfully!")
print("Dataset shape:", df_check.shape)
print("File location:", output_path.relative_to(PROJECT_ROOT))


Cleaned dataset saved successfully!
Dataset shape: (144, 18)
File location: data/cleaned/jps_coastal_erosion_cleaned_144.csv
